In [0]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Criar uma tabela de produtos já unindo categoria, marca (brands) e quantidade total     #   #   de estoque em todas as lojas                                                            #  
#    Origem camada bronze: brands                                                           #
#                          categories                                                       #
#                          products                                                         #
#                          stocks                                                           #
#    Destino camada silver: product                                                         #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela bronze                              #
# 2. Criar uma tabela Silver com os dados da tabela bronze                                  #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [0]:
import sys
import importlib

# 1. Garante que alterações no arquivo .py sejam carregadas automaticamente
%load_ext autoreload
%autoreload 2

# 2. Adiciona a pasta pai ('Codigo') com o prefixo '/Workspace'
import os; caminho_codigo = os.path.abspath("../..")
if caminho_codigo not in sys.path:
    sys.path.append(caminho_codigo)

# 3. Importa e força o recarregamento do módulo
import funcoes_diversas.funcoes_util
importlib.reload(funcoes_diversas.funcoes_util)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
mapa_criado = gerar_tabela_temporaria(["brands", "categories", "products","stocks"])

View 'tmp_bronze_brands' criada com sucesso!
View 'tmp_bronze_categories' criada com sucesso!
View 'tmp_bronze_products' criada com sucesso!
View 'tmp_bronze_stocks' criada com sucesso!


In [0]:
df_product_silver = spark.sql("""with stocks as(
    select product_id, 
       sum(quantity) as total_stocks 
       from tmp_bronze_stocks 
       group by product_id)
    select p.product_id,
           p.product_name,
           b.brand_name,
           c.category_name,
           p.model_year,
           p.list_price,
           s.total_stocks
    from tmp_bronze_products        as p
    left join tmp_bronze_categories as c on p.category_id = c.category_id
    left join tmp_bronze_brands     as b on p.brand_id    = b.brand_id
    left join stocks                as s on p.product_id  = s.product_id """)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8989934322844711>, line 1
----> 1 df_product_silver = spark.sql("""with stocks as(
      2     select product_id, 
      3        sum(quantity) as total_stocks 
      4        from tmp_bronze_stocks 
      5        group by product_id)
      6     select p.product_id,
      7            p.product_name,
      8            b.brand_name,
      9            c.category_name,
     10            p.model_year,
     11            p.list_price,
     12            s.total_stocks
     13     from tmp_bronze_products        as p
     14     left join tmp_bronze_categories as c on p.category_id = c.category_id
     15     left join tmp_bronze_brands     as b on p.brand_id    = b.brand_id
     16     left join stocks                as s on p.product_id  = s.product_id """)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql

In [0]:
silver_path = buscar_caminho("silver")
# transformando consulta em uma tabela Silver 
df_product_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{silver_path}/product")